In [1]:
 
import numpy as np 
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split

from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import math
import pandas as pd 
import matplotlib.pyplot as plt

# import seaborn as sns
import sklearn.metrics as metrics
%matplotlib inline
import os
from pandas_ml import ConfusionMatrix

In [2]:
data = pd.read_csv('CIC-IDS-2017-9.csv')
# test = pd.read_csv('UNSW_NB15_testing-set.csv')
# combined_data = pd.concat([train, test]).drop(['id'],axis=1)
combined_data = data

In [3]:
combined_data.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0.005989,0.939505,0.000149,0.000056,0.005220,8.063796e-06,0.016237,0.000000,0.033918,0.029041,...,0.571429,0.000003,3.124006e-07,0.000003,2.688679e-06,0.134167,0.006512,0.136667,0.128333,BENIGN
1,0.000000,0.947978,0.002616,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.088319,1.453303e-01,0.178302,1.792453e-07,0.101667,0.090546,0.173333,0.045875,BENIGN
2,0.082451,0.000834,0.000101,0.000000,0.000499,0.000000e+00,0.001128,0.012043,0.004713,0.000000,...,0.571429,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,BENIGN
3,0.000000,0.000456,0.000014,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,BENIGN
4,0.001355,0.000005,0.000029,0.000014,0.000392,6.602871e-07,0.009388,0.000000,0.011639,0.015883,...,0.357143,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,BENIGN


In [4]:
vector = combined_data[' Label']

In [5]:
from sklearn.preprocessing import LabelEncoder,normalize
le1 = LabelEncoder()
le = LabelEncoder()

vector = combined_data[' Label']
print("Label:", set(list(vector))) # use print to make it print on single line 

combined_data[' Label'] = le1.fit_transform(vector)
# combined_data['proto'] = le.fit_transform(combined_data['proto'])
# combined_data['service'] = le.fit_transform(combined_data['service'])
# combined_data['state'] = le.fit_transform(combined_data['state'])

vector = combined_data[' Label']

Label: {'BENIGN', 'Bot'}


In [6]:
# combined_data = data[~data['Label'].isin([2])] 

In [7]:
y_label = combined_data.iloc[:,78].values.flatten()
dict = {}
for i in y_label:
    dict.update({i:dict.get(i,0)+1})
dict

{0: 189067, 1: 1966}

In [8]:
le1.inverse_transform([0,1])
combined_data.head(3)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0.005989,0.939505,0.000149,0.000056,0.005220,0.000008,0.016237,0.000000,0.033918,0.029041,...,0.571429,0.000003,3.124006e-07,0.000003,2.688679e-06,0.134167,0.006512,0.136667,0.128333,0
1,0.000000,0.947978,0.002616,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.088319,1.453303e-01,0.178302,1.792453e-07,0.101667,0.090546,0.173333,0.045875,0
2,0.082451,0.000834,0.000101,0.000000,0.000499,0.000000,0.001128,0.012043,0.004713,0.000000,...,0.571429,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0


In [9]:
# combined_data_reduced=combined_data.drop(['Timestamp','Flow Duration'],axis=1)
combined_data_reduced=combined_data

combined_data_reduced = combined_data_reduced.dropna(axis=0)
combined_data_reduced.replace([np.inf, -np.inf], 0)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0.005989,9.395047e-01,0.000149,0.000056,0.005220,8.063796e-06,0.016237,0.000000,0.033918,0.029041,...,0.571429,0.000003,3.124006e-07,0.000003,2.688679e-06,0.134167,0.006512,0.136667,0.128333,0
1,0.000000,9.479782e-01,0.002616,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.088319,1.453303e-01,0.178302,1.792453e-07,0.101667,0.090546,0.173333,0.045875,0
2,0.082451,8.344833e-04,0.000101,0.000000,0.000499,0.000000e+00,0.001128,0.012043,0.004713,0.000000,...,0.571429,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0
3,0.000000,4.564333e-04,0.000014,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0
4,0.001355,5.241666e-06,0.000029,0.000014,0.000392,6.602871e-07,0.009388,0.000000,0.011639,0.015883,...,0.357143,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0
5,0.015874,1.666667e-07,0.000000,0.000004,0.000005,9.569378e-09,0.000242,0.002581,0.001010,0.000000,...,0.357143,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0
6,0.001355,7.441666e-06,0.000038,0.000014,0.000531,4.886762e-06,0.012611,0.000000,0.012269,0.019314,...,0.357143,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0
7,0.001355,8.900000e-06,0.000038,0.000021,0.002537,4.861244e-06,0.062530,0.000000,0.058615,0.096813,...,0.357143,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0
8,0.001355,4.391666e-06,0.000029,0.000014,0.002277,4.497608e-06,0.056285,0.000000,0.067619,0.096449,...,0.357143,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0
9,0.015920,2.500000e-07,0.000000,0.000004,0.000005,9.569378e-09,0.000242,0.002581,0.001010,0.000000,...,0.357143,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0


In [10]:

print(np.isnan(combined_data_reduced).any()[np.isnan(combined_data_reduced).any() == True])
print(np.isinf(combined_data_reduced).any()[np.isinf(combined_data_reduced).any() == True])

Series([], dtype: bool)
Series([], dtype: bool)


In [11]:
combined_data_reduced.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0.005989,0.939505,0.000149,0.000056,0.005220,8.063796e-06,0.016237,0.000000,0.033918,0.029041,...,0.571429,0.000003,3.124006e-07,0.000003,2.688679e-06,0.134167,0.006512,0.136667,0.128333,0
1,0.000000,0.947978,0.002616,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.088319,1.453303e-01,0.178302,1.792453e-07,0.101667,0.090546,0.173333,0.045875,0
2,0.082451,0.000834,0.000101,0.000000,0.000499,0.000000e+00,0.001128,0.012043,0.004713,0.000000,...,0.571429,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0
3,0.000000,0.000456,0.000014,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0
4,0.001355,0.000005,0.000029,0.000014,0.000392,6.602871e-07,0.009388,0.000000,0.011639,0.015883,...,0.357143,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0


In [12]:
data_x = combined_data_reduced.drop([' Label'], axis=1) # droped label
data_y = combined_data_reduced.loc[:,[' Label']]
# del combined_data # free mem
X_train, X_test, y_train, y_test = train_test_split(data_x, data_y, test_size=.20, random_state=42) # TODO

y_train = y_train.values.flatten()
y_test = y_test.values.flatten()

In [13]:
data_x2 = combined_data_reduced
data_y2 = combined_data_reduced.loc[:,[' Label']]

X_train2, X_test2, y_train2, y_test2 = train_test_split(data_x2, data_y2, test_size=0.2, random_state=42)


In [14]:
X_train.shape
# data_y2.max()

(152728, 78)

In [15]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape) # test is larger... good 
print(y_test.shape)
# y_train.min()

(152728, 78)
(152728,)
(38182, 78)
(38182,)


In [16]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error)
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import Normalizer

In [17]:
X = X_train
T = X_test
Y = y_train
C = y_test

# trainX = X
# testT = T
# trainlabel = Y
# testlabel =C


scaler = Normalizer().fit(X)
trainX = scaler.transform(X)

scaler = Normalizer().fit(T)
testT = scaler.transform(T)

traindata = np.array(trainX)
trainlabel = np.array(Y)

testdata = np.array(testT)
testlabel = np.array(C)
testlabel = testlabel.flatten()

In [18]:
testlabel.shape

(38182,)

In [19]:

KNN = KNeighborsClassifier()
KNN.fit(traindata, trainlabel)

DT = DecisionTreeClassifier()
DT.fit(traindata, trainlabel)

RF = RandomForestClassifier(n_estimators=100)
RF.fit(traindata, trainlabel)

RandomForestClassifier(bootstrap=True, class_weight=None, criterion='gini',
                       max_depth=None, max_features='auto', max_leaf_nodes=None,
                       min_impurity_decrease=0.0, min_impurity_split=None,
                       min_samples_leaf=1, min_samples_split=2,
                       min_weight_fraction_leaf=0.0, n_estimators=100,
                       n_jobs=None, oob_score=False, random_state=None,
                       verbose=0, warm_start=False)

In [20]:
np.seterr(invalid='ignore')

{'divide': 'warn', 'invalid': 'warn', 'over': 'warn', 'under': 'ignore'}

In [21]:

nums = 2  #分块数量

In [22]:
#-*- coding: utf8
from __future__ import division, print_function

import numpy as np

def _compute_centroids(X, assign, num_clusters):
    C = np.zeros(shape=(num_clusters, X.shape[1]), dtype='d')
    for k in range(num_clusters):

        if not (assign == k).any():
            continue

        K = X[assign == k]
        if K.ndim == 1:
            K = K[np.newaxis]
        C[k] = X[assign == k].mean(axis=0) 
    return C

def _surprisal_mat(X):
    #Some elements have zero prob, ingore and treat warnings
    with np.errstate(divide='ignore', invalid='ignore'):
        L = np.log2(X)
        L[np.isnan(L)] = 0
        L[np.isinf(L)] = 0
    return L

def _dist_all(X, C):
    S_x = _surprisal_mat(X)
    S_c = _surprisal_mat(C)
    
    D = (X * (S_x - S_c[:, np.newaxis,: ])).sum(axis=2).T
    return D

def _base_kmeans(X, C, n_iters=-1):
    
    num_clusters = C.shape[0]
    n = X.shape[0]

    C_final = C

    #KMeans algorithm
    cent_dists = None
    assign = None
    prev_assign = None
    best_shift = None

    iters = n_iters
    converged = False

    while iters != 0 and not converged:
        #assign elements to new clusters    
        D = _dist_all(X, C)
        assign = D.argmin(axis=1)
        
        #check if converged, if not compute new centroids
        if prev_assign is not None and not (prev_assign - assign).any():
            converged = True
        else: 
            C_final = _compute_centroids(X, assign, num_clusters)

        prev_assign = assign
        iters -= 1
    
    return C_final, assign

def cost(X, C, assign):
    cost = 0
    for k in set(assign):
        idx = assign == k
        cost += _dist_all(X[idx], C[k][np.newaxis]).sum()
    return cost

def klkmeans(X, num_clusters, n_iters=-1, n_runs=10):

    min_cost = float('+inf')
    best_C = None
    best_assign = None

    for _ in range(n_runs):
        assign = np.random.randint(0, num_clusters, X.shape[0])
        C = _compute_centroids(X, assign, num_clusters)

        C, assign = _base_kmeans(X, C, n_iters)
        clust_cost = cost(X, C, assign)

        if clust_cost < min_cost:
            best_C = C
            best_assign = assign

    return best_C, best_assign

if __name__ == '__main__':
    np.seterr(all='raise')
#     X = np.zeros((200, 1000))
#     X[0:100] = 1
#     X[100:200, 500:] = 1
#     X += 1e-20
    
#     X = (X.T / X.sum(axis=1)).T
#     C, assign = klkmeans(X, 2)
    
#     assert ((C.sum(axis=1) - 1) < 1e-10).all()
#     assert (assign[0:100] != assign[100:]).all()
    
    import os
#     dir_ = os.path.dirname('D:\mycode\CycleGAN_MetaLearning\src1_NSL-KDD')
#     fpath = os.path.dirname( './testdata.dat')
#     fpath = './testdata.dat'
#     X = np.genfromtxt(fpath)


    X = data_x2.values
#     X2 = X_test2.values
    
    
    C, assign = klkmeans(X, nums)
#     C2, assign2 = klkmeans(X2, nums)
#     assert ((C.sum(axis=1) - 1) < 1e-10).all()   
    assert len(set(assign)) == nums
#     assert len(set(assign2)) == nums
  
#     for nums in range(10): 
#         nums += 1        
#         C, assign = klkmeans(X, nums)
##         C2, assign2 = klkmeans(X2, nums)
#         print(nums)
#         if len(set(assign)) == nums :
#             print("----"+str(nums))

In [23]:
assign_ = assign.reshape(len(assign),1)
X_ = np.concatenate((X,assign_),axis = 1)

# assign2_ = assign2.reshape(len(assign2),1)
# X2_ = np.concatenate((X2,assign2_),axis = 1)

In [24]:
assign

array([1, 1, 1, ..., 0, 1, 1], dtype=int64)

In [25]:
print(X_.shape)
# print(X2_.shape)

(190910, 80)


In [26]:
CC={}
CT={}
for i in range(nums):
    
    X_df = X_[X_[...,-1]==i,:-1]
    y_df= X_[X_[...,-1]==i,-1]

    CC[i], CT[i], y_tr, y_te = train_test_split(X_df, y_df, test_size=0.2, random_state=42)
    print(CC[i].shape,end = " ")
    print(CT[i].shape)

(32220, 79) (8055, 79)
(120508, 79) (30127, 79)


In [27]:
# CC={}
# for i in range(nums):
#     CC[i] = X_[X_[...,-1]==i,:-1]
#     print(CC[i].shape)
  
# print()
# CT={}
# for i in range(nums):
#     CT[i] = X2_[X2_[...,-1]==i,:-1]
#     print(CT[i].shape)

In [28]:
# np.set_printoptions(suppress=True)
# list = np.zeros(4)
# for i in assign:
#     list[i] += 1
# list

In [29]:
CC[i][...,-1].shape

(120508,)

In [30]:
knn = []
dt = []
rf = []
for i in range(nums):
    knn_ = KNeighborsClassifier()
    knn_.fit(CC[i][...,:-1], CC[i][...,-1].flatten())
    knn.append(knn_)
    
    dt_ = DecisionTreeClassifier()
    dt_.fit(CC[i][...,:-1], CC[i][...,-1].flatten())
    dt.append(dt_)

    rf_ = RandomForestClassifier(n_estimators=100)
    rf_.fit(CC[i][...,:-1], CC[i][...,-1].flatten())
    rf.append(rf_)
    
#     expected = CT[i][...,-1].flatten()
#     predicted = knn_.predict(CT[i][...,:-1])
#     # summarize the fit of the model
#     cm = ConfusionMatrix(expected, predicted)
#     print(cm)
#     try:
# #         np.errstate(divide="ignore")
#         cm.stats()
#     except:
#         continue
# #     print(cm.to_dataframe().loc['actual' = 'False', 'predicted' = 'False'])
#     print(cm(super.get(actual = 'False', predicted = 'False'))

In [31]:
# k= 3
# assign = np.random.randint(0, 4, 10)
# idx = assign == k
# print(idx)
# print(assign)


In [32]:
from pymoo.core.problem import Problem
class MyProblem(Problem):
    def __init__(self):
        self.i = i
        self.P1 = P1
        self.P2 = P2
        self.P3 = P3
        self.TP1 = TP1
        self.TP2 = TP2
        self.TP3 = TP3
        self.TN1 = TN1
        self.TN2 = TN2
        self.TN3 = TN3
        super().__init__(n_var=3,   # 变量数
                         n_obj=1,   # 目标数
                         n_constr=2,    # 约束数
                         xl=np.array([0,0,0]),     # 变量下界
                         xu=np.array([1,1,1]),   # 变量上界
                         )

    def _evaluate(self, x, out, *args, **kwargs):

        # 定义目标函数
#         f = dict([(key,[]) for key in range(4)])
#         f = dict.fromkeys(range(0,4),[])
#         print(i+1)
        
        f = 1 - (TP1 + TN1) * x[:,0] / P1  - (TP2 + TN2) * x[:,1] / P2 - (TP3 + TN3) * x[:,2] / P3 
    # 定义约束条件
        g1 = x[:,0] + x[:,1] + x[:,2] - 1
        g2 = - x[:,0] - x[:,1] - x[:,2] + 0.9
        # todo
        out["F"] = np.column_stack([f])
        out["G"] = np.column_stack([g1,g2])
        
#         print(cm1)
#         print(cm2)
#         print(cm3)
#         print()



In [33]:
# pd.crosstab(cm1._y_true, cm1._y_pred).reindex()[1][1]
# pd.crosstab(cm1._y_true, cm1._y_pred).iloc[1][1]

In [34]:

from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.factory import get_sampling, get_crossover, get_mutation
from pymoo.optimize import minimize
# from example import MyProblem

# 定义遗传算法
algorithm = NSGA2(
    pop_size=40,
    n_offsprings=10,
    sampling=get_sampling("real_random"),
    crossover=get_crossover("real_sbx", prob=0.9, eta=15),
    mutation=get_mutation("real_pm", eta=20),
    eliminate_duplicates=True
)



Compiled modules for significant speedup can not be used!
https://pymoo.org/installation.html#installation

To disable this warning:
from pymoo.config import Config
Config.show_compile_hint = False



In [35]:
ans = []
for i in range(nums):
    print(i)
    
    predicted1 = knn[i].predict(CC[i][...,:-1])
    predicted2 = dt[i].predict(CC[i][...,:-1])
    predicted3 = rf[i].predict(CC[i][...,:-1])
    cm1 = ConfusionMatrix(CC[i][...,-1], predicted1)
    cm2 = ConfusionMatrix(CC[i][...,-1], predicted2)
    cm3 = ConfusionMatrix(CC[i][...,-1], predicted3)

    a = pd.crosstab(cm1._y_true, cm1._y_pred)
    TP1 = a.reindex()[0][0]
    TN1 = a.reindex()[1][1]
    P1 = cm1.population

    b = pd.crosstab(cm2._y_true, cm2._y_pred)
    TP2 = b.reindex()[0][0]
    TN2 = b.reindex()[1][1]
    P2 = cm2.population

    c = pd.crosstab(cm3._y_true, cm3._y_pred)
    TP3 = c.reindex()[0][0]
    TN3 = c.reindex()[1][1]
    P3= cm3.population
    print(cm1)
    print(cm2)
    print(cm3)
    print()
    
    res = minimize(MyProblem(),
                   algorithm,
                   ('n_gen', 40),
                   seed=1,
                   verbose=False
                   )
    ans.append(res)
    


0
Predicted  False  True  __all__
Actual                         
False      30690    20    30710
True         478  1032     1510
__all__    31168  1052    32220
Predicted  False  True  __all__
Actual                         
False      30708     2    30710
True          53  1457     1510
__all__    30761  1459    32220
Predicted  False  True  __all__
Actual                         
False      30710     0    30710
True         352  1158     1510
__all__    31062  1158    32220

1
Predicted   False  True  __all__
Actual                          
False      120420     0   120420
True           87     1       88
__all__    120507     1   120508
Predicted   False  True  __all__
Actual                          
False      120420     0   120420
True            3    85       88
__all__    120423    85   120508
Predicted   False  True  __all__
Actual                          
False      120420     0   120420
True           14    74       88
__all__    120434    74   120508



In [36]:
for j in range(nums):
    
    print(ans[j].X)

[0.41417927 0.04995346 0.53583676]
[0.41417927 0.04995346 0.53583676]


In [37]:
ans[1].X[1]

0.04995345894608716

In [38]:
expected = testlabel
np.savetxt("Exp.txt", expected) 

predicted1 = KNN.predict(testdata)
predicted2 =  DT.predict(testdata)
predicted3 = RF.predict(testdata)

cm1 = ConfusionMatrix(expected, predicted1)
np.savetxt("Pre1.txt", predicted1) 

cm2 = ConfusionMatrix(expected, predicted2)
np.savetxt("Pre2.txt", predicted2) 

cm3 = ConfusionMatrix(expected, predicted3)
np.savetxt("Pre3.txt", predicted3) 

cm1.stats()

OrderedDict([('population', 38182),
             ('P', 395),
             ('N', 37787),
             ('PositiveTest', 7),
             ('NegativeTest', 38175),
             ('TP', 1),
             ('TN', 37781),
             ('FP', 6),
             ('FN', 394),
             ('TPR', 0.002531645569620253),
             ('TNR', 0.999841215232752),
             ('PPV', 0.14285714285714285),
             ('NPV', 0.9896791093647676),
             ('FPR', 0.00015878476724799534),
             ('FDR', 0.8571428571428571),
             ('FNR', 0.9974683544303797),
             ('ACC', 0.9895238594101933),
             ('F1_score', 0.004975124378109453),
             ('MCC', 0.01773386810007039),
             ('informedness', 0.002372860802372312),
             ('markedness', 0.13253625222191046),
             ('prevalence', 0.010345188832434132),
             ('LRP', 15.943881856540084),
             ('LRN', 0.997626762363642),
             ('DOR', 15.98181049069374),
             ('FOR', 0.010

In [39]:
cm2.stats()

OrderedDict([('population', 38182),
             ('P', 395),
             ('N', 37787),
             ('PositiveTest', 444),
             ('NegativeTest', 37738),
             ('TP', 23),
             ('TN', 37366),
             ('FP', 421),
             ('FN', 372),
             ('TPR', 0.05822784810126582),
             ('TNR', 0.9888586021647656),
             ('PPV', 0.0518018018018018),
             ('NPV', 0.9901425618739732),
             ('FPR', 0.011141397835234339),
             ('FDR', 0.9481981981981982),
             ('FNR', 0.9417721518987342),
             ('ACC', 0.9792310512807082),
             ('F1_score', 0.054827175208581644),
             ('MCC', 0.044441098030536065),
             ('informedness', 0.04708645026603153),
             ('markedness', 0.04194436367577503),
             ('prevalence', 0.010345188832434132),
             ('LRP', 5.226260561051144),
             ('LRN', 0.9523830301289266),
             ('DOR', 5.4875616172451664),
             ('FOR', 0.

In [40]:
cm3.stats()

OrderedDict([('population', 38182),
             ('P', 395),
             ('N', 37787),
             ('PositiveTest', 74),
             ('NegativeTest', 38108),
             ('TP', 4),
             ('TN', 37717),
             ('FP', 70),
             ('FN', 391),
             ('TPR', 0.010126582278481013),
             ('TNR', 0.9981475110487734),
             ('PPV', 0.05405405405405406),
             ('NPV', 0.9897396872047864),
             ('FPR', 0.0018524889512266122),
             ('FDR', 0.9459459459459459),
             ('FNR', 0.9898734177215189),
             ('ACC', 0.9879262479702478),
             ('F1_score', 0.017057569296375266),
             ('MCC', 0.019035585158467743),
             ('informedness', 0.008274093327254484),
             ('markedness', 0.0437937412588405),
             ('prevalence', 0.010345188832434132),
             ('LRP', 5.466473779385172),
             ('LRN', 0.9917105505592447),
             ('DOR', 5.5121666057727445),
             ('FOR', 0.

In [41]:
expectedEnd = np.empty(shape = [0,1]).flatten()
predictedEnd = np.empty(shape = [0,1]).flatten()
for i in range(nums):
    expected2 = CT[i][...,-1]
    testdata2 = CT[i][...,:-1]
    predicted2 = np.around(ans[i].X[0] * knn[i].predict(testdata2) +  ans[i].X[1] * dt[i].predict(testdata2) + ans[i].X[2] * rf[i].predict(testdata2))
#     predicted2 = np.around(0.09 * knn[i].predict(testdata2) +  0.3 * dt[i].predict(testdata2) + 0.2 * rf[i].predict(testdata2))
    expectedEnd = np.concatenate((expectedEnd, expected2), axis = 0)
    predictedEnd = np.concatenate((predictedEnd,predicted2), axis = 0)
    
    print(expectedEnd.shape)
    print(predictedEnd.shape)

cm = ConfusionMatrix(expectedEnd, predictedEnd)
expected = np.array(expectedEnd)
predicted = np.array(predictedEnd)

np.savetxt("AdassPre.txt", predicted) 
np.savetxt("AdassExp.txt", expected) 

cm.stats()


(8055,)
(8055,)
(38182,)
(38182,)


OrderedDict([('population', 38182),
             ('P', 366),
             ('N', 37816),
             ('PositiveTest', 226),
             ('NegativeTest', 37956),
             ('TP', 214),
             ('TN', 37804),
             ('FP', 12),
             ('FN', 152),
             ('TPR', 0.5846994535519126),
             ('TNR', 0.9996826740004231),
             ('PPV', 0.9469026548672567),
             ('NPV', 0.9959953630519549),
             ('FPR', 0.00031732599957689866),
             ('FDR', 0.05309734513274336),
             ('FNR', 0.41530054644808745),
             ('ACC', 0.9957047823581793),
             ('F1_score', 0.722972972972973),
             ('MCC', 0.7423023304399018),
             ('informedness', 0.5843821275523355),
             ('markedness', 0.9428980179192115),
             ('prevalence', 0.009585668639673145),
             ('LRP', 1842.5828779599271),
             ('LRN', 0.41543237394140503),
             ('DOR', 4435.337719298245),
             ('FOR', 0.004